In [0]:
%pip install yfinance arch
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
from arch import arch_model

In [0]:
spy = yf.Ticker("SPY").history(start="2005-01-01", auto_adjust=True)
vix = yf.Ticker("^VIX").history(start="2005-01-01")["Close"].rename("VIX")

# strip timezones so the two indexes line up on date
spy.index = spy.index.tz_localize(None).normalize()
vix.index = vix.index.tz_localize(None).normalize()

df = spy[["Open", "High", "Low", "Close"]].join(vix, how="inner").dropna()
print(df.shape)
df.tail()

In [0]:
o, h, l, c = (np.log(df[x]) for x in ["Open", "High", "Low", "Close"])

overnight = o - c.shift(1)                    # gap from yesterday's close to today's open
df["rv"] = overnight**2 + 0.5*(h - l)**2 - (2*np.log(2) - 1)*(c - o)**2
df["ret"] = 100 * (c - c.shift(1))            # daily log return in percent (for GARCH)

df = df.dropna()
df[["rv", "ret"]].describe()

In [0]:
df["rv_d"] = df["rv"]                          # yesterday's variance (daily)
df["rv_w"] = df["rv"].rolling(5).mean()        # last week's average
df["rv_m"] = df["rv"].rolling(22).mean()       # last month's average
df["iv"]   = (df["VIX"] / 100)**2 / 252        # VIX converted to daily implied variance
df["target"] = df["rv"].shift(-1)              # TOMORROW's variance, what we predict

data = df.dropna()
train = data[:"2018-12-31"]
test  = data["2019-01-01":]
print(len(train), len(test))

In [0]:
def fit_har(cols):
    X_train = sm.add_constant(train[cols])
    model = sm.OLS(train["target"], X_train).fit(cov_type="HAC", cov_kwds={"maxlags": 22})
    preds = model.predict(sm.add_constant(test[cols])).clip(lower=1e-10)
    return model, preds

har, har_pred = fit_har(["rv_d", "rv_w", "rv_m"])
har_iv, har_iv_pred = fit_har(["rv_d", "rv_w", "rv_m", "iv"])

print(har.summary().tables[1])
print(har_iv.summary().tables[1])

preds = pd.DataFrame({"HAR-RV": har_pred, "HAR-RV-IV": har_iv_pred}, index=test.index)

In [0]:
am = arch_model(df["ret"], mean="Constant", vol="GARCH", p=1, q=1, dist="t")
garch = am.fit(last_obs=test.index[0], disp="off")
print(garch.summary())

fc = garch.forecast(horizon=1, start=test.index[0], reindex=False)
preds["GARCH"] = (fc.variance["h.1"] / 1e4).reindex(test.index)

In [0]:
def fit_log_har(cols):
    Xtr = sm.add_constant(np.log(train[cols].clip(lower=1e-10)))
    ytr = np.log(train["target"].clip(lower=1e-10))
    model = sm.OLS(ytr, Xtr).fit(cov_type="HAC", cov_kwds={"maxlags": 22})
    Xte = sm.add_constant(np.log(test[cols].clip(lower=1e-10)))
    # exp(mean of log) underestimates the mean, so add half the residual variance
    return model, np.exp(model.predict(Xte) + 0.5 * model.mse_resid)

log_har, preds["log-HAR-RV"] = fit_log_har(["rv_d", "rv_w", "rv_m"])
log_har_iv, preds["log-HAR-RV-IV"] = fit_log_har(["rv_d", "rv_w", "rv_m", "iv"])
print(log_har_iv.summary().tables[1])

In [0]:
y = test["target"].clip(lower=1e-10)
preds = preds.dropna()
y = y.loc[preds.index]

def qlike(actual, forecast):
    ratio = actual / forecast
    return np.mean(ratio - np.log(ratio) - 1)

results = pd.DataFrame({
    m: {"MSE (x1e8)": np.mean((y - preds[m])**2) * 1e8,
        "QLIKE": qlike(y, preds[m])}
    for m in preds.columns
}).T.sort_values("QLIKE")
results

In [0]:
ann = lambda v: np.sqrt(252 * v) * 100      # daily variance -> annualized vol %

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ann(y).rolling(5).mean(), color="lightgray", lw=2, label="Realized (5-day avg)")
for m in preds.columns:
    ax.plot(ann(preds[m]), lw=1, label=m)
ax.set_ylabel("Annualized volatility (%)")
ax.set_title("SPY one-day-ahead volatility forecasts, out of sample 2019-2026")
ax.legend()
plt.show()

In [0]:
out = preds.join(y.rename("realized")).reset_index().rename(columns={"index": "date"})
out.columns = [c.replace("-", "_") for c in out.columns]   # cleaner column names for SQL

(spark.createDataFrame(out)
     .write.mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("spx_vol_forecasts"))